Steps to preprocess MODIS data for phenology tracking. These methods are 100% adapted from Jerod Merkle and Shannon Albeke's workflow to process data in R. 

Steps 1a, 1b and the first function in step 2 (ndviCALC_clean) all run within GEE (~10 seconds). However, GEE runs into memory issues when trying to fit the curves on the image collections (the spatial data that is stored in GEE). So, we export the image collections as rasters and then will read it back in.

Conversion to Python and GEE by Tayler LaSharr

October 2024

In [20]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import numpy as np
from scipy.optimize import curve_fit # for fitting the DLC curves (similar to NLS in R)
import multiprocessing as mp


## if you need to authenticate GEE on your machine, use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module (change 'tlasharr' to your project name)
ee.Initialize(project='ee-tlasharr')

Step 1a. Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [21]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_winter_month_day = '11-01'
end_winter_month_day = '03-01'

## get the time period we care about
start_date_data = '2001-01-01'
end_date_data = '2012-12-31'

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
# bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])
# bounding_box = ee.Geometry.Rectangle([-111.715449, 44.160084, -111.521152, 44.26045])

## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )
modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))

## save target projection and scale
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## just check on a couple of things
print('Projection of MODIS NDVI data:', modis_projection.getInfo())
print('Reprojected Bounding Box:', reprojected_bounding_box.getInfo())
print("Bands available in the first image:", modis_ndvi.first().bandNames().getInfo())
print("Target projection:", target_projection)
print("Target scale:", target_scale)


Projection of MODIS NDVI data: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Reprojected Bounding Box: {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459407], [-9135043.795726, 4686382.625733], [-9195643.896612, 4639491.792059001]]]}
Bands available in the first image: ['sur_refl_b01', 'sur_refl_b02', 'QA', 'State', 'NDVI']
Target projection: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Target scale: 231.65635826

Step 1b. Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function: removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

Second function: creates a Snow Free Mask. This function will create an image collection called 'SnowCover' that has an image with pixels set to 1 (snow) or 0 (no snow) for each image during the time period you are interested in. This is the equivalent of the 'SnowCover' rasters produced in step 1b of the R code.

In [22]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    is_land = state.rightShift(3).bitwiseAnd(7).eq(1)  ## land (not in Jerod's code)
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice).And(is_land)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information from the State band of the MODIS data
def extract_snow(image):
    
    ## get the state band
    state = image.select('State')

    ## bit 12 represents snow/ice
    snow_mod35 = state.bitwiseAnd(1 << 12).neq(0)  # 1 if snow/ice is present, 0 otherwise

    ## bit 16 represents internal snow
    snow_internal = state.bitwiseAnd(1 << 16).neq(0)  # 1 if internal snow is present, 0 otherwise

    ## combine the MOD35 and internal snow
    snow_combined = snow_mod35.Or(snow_internal)  # 1 if either MOD35 or internal snow is present

    ## add the snow band to the image
    snow_band = snow_combined.rename('SnowCovered')

    ## carry over the system:time_start property from the original image (this is not saved in the State band, so you need to explicitly call it back in)
    ## and make sure it stays in the native projection

    snow_band = snow_band.set('system:time_start', image.get('system:time_start'))
    return snow_band


## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = modis_ndvi.map(extract_snow)

## just check on a couple of things
print('Filtered MODIS information:', NDVIFiltered.getInfo())
print("Bands available in the first image:", NDVIFiltered.first().bandNames().getInfo())

print('Filtered MODIS Snow information (NDVI):', SnowCover.getInfo())
print("Bands available in the first image (Snow):", SnowCover.first().bandNames().getInfo())



Filtered MODIS information: {'type': 'ImageCollection', 'bands': [], 'version': 1731611857325943, 'id': 'MODIS/061/MOD09Q1', 'properties': {'system:is_global': 1}, 'features': [{'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}], 'version': 1690253155865877, 'id': 'MODIS/061/MOD09Q1/2001_01_01', 'properties': {'system:footprint': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459

In [30]:
## functions to check the projection at each step
def print_projection_info(collection, step_name):
    # Get the first image in the collection
    first_image = collection.first()
    if not isinstance(first_image, ee.Image):
        raise TypeError(f"First element in {step_name} is not an ee.Image")

    # Get the projection details
    projection = first_image.select(0).projection()  # Ensure a band is selected
    crs = projection.crs().getInfo()  # Get the CRS
    scale = projection.nominalScale().getInfo()  # Get the scale in meters

    print(f"{step_name} Projection Information:")
    print(f" - CRS: {crs}")
    print(f" - Scale: {scale} meters")


def print_image_projection(image, name="Single Image"):
    # Get the projection of the image
    projection = image.projection()
    
    # Retrieve CRS and scale
    crs = projection.crs().getInfo()
    scale = projection.nominalScale().getInfo()

    # Print the results
    print(f"{name} Projection:")
    print(f"  CRS: {crs}")
    print(f"  Scale: {scale} meters\n")

## function to split area into tiles for faster processing
def split_region_into_tiles(region, tile_size):
    
    # Get the bounding box of the region
    bounds = region.bounds().coordinates().get(0)
    bounds = ee.List(bounds)

    # Extract coordinates
    min_lon = ee.Number(ee.List(bounds.get(0)).get(0))
    min_lat = ee.Number(ee.List(bounds.get(0)).get(1))
    max_lon = ee.Number(ee.List(bounds.get(2)).get(0))
    max_lat = ee.Number(ee.List(bounds.get(2)).get(1))

    # Generate ranges for longitude and latitude
    lon_range = ee.List.sequence(min_lon, max_lon, tile_size)
    lat_range = ee.List.sequence(min_lat, max_lat, tile_size)

    # Create grid of tiles
    tiles = []
    lon_count = lon_range.size().subtract(1)
    lat_count = lat_range.size().subtract(1)

    for lon_index in range(lon_count.getInfo()):
        for lat_index in range(lat_count.getInfo()):
            lon_min = lon_range.get(lon_index)
            lon_max = lon_range.get(ee.Number(lon_index).add(1))
            lat_min = lat_range.get(lat_index)
            lat_max = lat_range.get(ee.Number(lat_index).add(1))

            # Append tile geometry
            tiles.append(
                ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
            )

    return tiles



Step 2. Calculations - the following functions go through 7/8 of the main steps in the ndviCALC_clean function from R. I was running into memory issues with GEE when I tried scaling the data, so I exported after step 7 and will read back in the data as a raster to finish the remainder of the steps.

ndviCALC_clean:

Perform all steps in the ndviCALC_clean function on the filtered MODIS data, with the exception of scaling the data. I ran into a lot of issues trying to scale the data in GEE, so resorted to exporting it and reading it back in as a GeoTiff/raster in Python.

In [24]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):
    
    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.gte(0).And(image.lte(2)))
    
    NDVICollection = NDVICollection.map(reclassify_outliers)
    return NDVICollection

## step 2: reclassify snow values during winter as 2
def ndviCALC_step2(SnowCollection, start_winter_month_day='11-01', end_winter_month_day='03-30', target_projection=None):

    ## split the winter start and end dates into months and days
    start_winter_month, start_winter_day = map(int, start_winter_month_day.split('-'))
    end_winter_month, end_winter_day = map(int, end_winter_month_day.split('-'))

    ## function to classify whether the date is in the winter period
    def classify_winter(image):
        ## extract the date of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_winter_month, start_winter_day)
        winter_end = ee.Date.fromYMD(year, end_winter_month, end_winter_day)

        ## use date difference to identify if the image falls within winter
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## mask the image during winter (i.e., replace NDVI values with 2 during winter)
        image_winter_adjusted = image.where(is_in_winter, ee.Image.constant(2))

        if target_projection:
            image_winter_adjusted = image_winter_adjusted.reproject(crs=target_projection, scale=target_projection.nominalScale())

        return image_winter_adjusted.copyProperties(image, image.propertyNames())

    ## map the function over the collection
    SnowCollection_winter_adjusted = SnowCollection.map(classify_winter)
    return SnowCollection_winter_adjusted

## step 3: check if pixel has been snow-free for 3 consecutive images
def ndviCALC_step3(SnowCollection):

    snow_list = SnowCollection.toList(SnowCollection.size())
    
    def process_snow(index):
        current = ee.Image(snow_list.get(index))
        previous1 = ee.Image(snow_list.get(index.subtract(1))).unmask(1)
        previous2 = ee.Image(snow_list.get(index.subtract(2))).unmask(1)
        
        snow_free = current.add(previous1).add(previous2).eq(0)
        return current.where(snow_free.Not(), 2)
    
    def iterate_func(i, result):
        i = ee.Number(i)
        result = ee.ImageCollection(result)
        processed = ee.Algorithms.If(
            i.gte(2), 
            process_snow(i),
            snow_list.get(i)
        )
        return result.merge(ee.ImageCollection([processed]))
    
    # Initialize the result collection with the first two images
    initial = ee.ImageCollection([snow_list.get(0), snow_list.get(1)])
    adjusted_snow = ee.List.sequence(2, SnowCollection.size().subtract(1)).iterate(iterate_func, initial)
    return ee.ImageCollection(adjusted_snow)


In [25]:
#### run all steps sequentially ####
NDVIFiltered = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")
print_projection_info(NDVIFiltered, "Processed NDVI (Step 1)")

SnowCover = ndviCALC_step2(SnowCover, start_winter_month_day='11-01', end_winter_month_day='03-30', target_projection=modis_projection)
print("Step 2 complete: Reclassified snow values during winter.")
print_projection_info(SnowCover, "Snow Cover (Step 2)")

SnowCover = ndviCALC_step3(SnowCover)
print("Step 3 complete: Checked if pixels have been snow-free for 3 consecutive images.")
print_projection_info(SnowCover, "Snow Cover (Step 3)")

Step 1 complete: Removed outliers from data.
Processed NDVI (Step 1) Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters

Step 2 complete: Reclassified snow values during winter.
Snow Cover (Step 2) Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters

Step 3 complete: Checked if pixels have been snow-free for 3 consecutive images.
Snow Cover (Step 3) Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters



In [26]:

## step 4 : mask pixels that are in winter or covered by snow and apply median filter
def ndviCALC_step4(image_collection, snow_collection, window_size, region, tile_size, target_projection=None):
    
    # Get the tiles for the region
    tiles = split_region_into_tiles(region, tile_size)

    # Process each tile for masking and filtering
    def process_tile(tile):
        tile_ndvi = image_collection.map(lambda img: img.clip(tile))
        tile_snow = snow_collection.map(lambda img: img.clip(tile))

        # Temporarily mask snow-covered pixels
        def preprocess_for_filter(ndvi_image):
            snow_image = tile_snow.filterDate(ndvi_image.date(), ndvi_image.date()).first()
            snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))
            return ndvi_image.updateMask(snow_image.Not())

        preprocessed_collection = tile_ndvi.map(preprocess_for_filter)

        # Apply median filter
        def smooth_image(current_image):
            all_images = preprocessed_collection.sort('system:time_start')
            all_images_list = all_images.toList(all_images.size())

            image_index = all_images_list.indexOf(current_image)
            half_window = ee.Number(window_size // 2)

            start_index = ee.Number(image_index).subtract(half_window).max(0)
            end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))
            window_images = all_images_list.slice(start_index, end_index.add(1))
            window_collection = ee.ImageCollection.fromImages(window_images)

            median_image = window_collection.median().toFloat()
            if target_projection:
                median_image = median_image.reproject(crs=target_projection, scale=target_projection.nominalScale())
            return median_image.copyProperties(current_image, current_image.propertyNames())

        smoothed_collection = preprocessed_collection.map(smooth_image)

        # Reapply snow mask
        def postprocess_after_filter(ndvi_image):
            snow_image = tile_snow.filterDate(ndvi_image.date(), ndvi_image.date()).first()
            snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))
            return ndvi_image.updateMask(snow_image.Not())

        final_tile_collection = smoothed_collection.map(postprocess_after_filter)
        return final_tile_collection

    # Process all tiles and merge results
    tiled_results = [process_tile(tile) for tile in tiles]
    merged_collection = ee.ImageCollection(tiled_results).flatten()
    return merged_collection


## step 5: calculate the floor (0.025 quantile)
def ndviCALC_step5(image_collection, quantile_value, region, tile_size):
    tiles = split_region_into_tiles(region, tile_size)

    def process_tile(tile):
        tile_ndvi = image_collection.map(lambda img: img.clip(tile))
        reference_projection = tile_ndvi.first().projection()

        # Calculate the quantile for the tile
        baseline_quantile = tile_ndvi.reduce(ee.Reducer.percentile([quantile_value]))
        floored_baseline_image = baseline_quantile.rename('NDVI_baseline')

        floored_baseline_image = floored_baseline_image.reproject(
            crs=reference_projection.crs(),
            scale=reference_projection.nominalScale()
        )
        return floored_baseline_image

    tiled_results = [process_tile(tile) for tile in tiles]
    return ee.ImageCollection(tiled_results).mosaic()

## step 7: mask all NDVI values based on the SnowCollection
def ndviCALC_step6(image_collection, snow_collection, region, tile_size):
    tiles = split_region_into_tiles(region, tile_size)

    def process_tile(tile):
        tile_ndvi = image_collection.map(lambda img: img.clip(tile))
        tile_snow = snow_collection.map(lambda img: img.clip(tile))

        def mask_ndvi_with_snow(ndvi_image):
            snow_image = tile_snow.filterDate(ndvi_image.date(), ndvi_image.date()).first()
            snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))
            snow_mask = snow_image.eq(1)  # Snow-covered pixels
            return ndvi_image.updateMask(snow_mask.Not())

        return tile_ndvi.map(mask_ndvi_with_snow)

    tiled_results = [process_tile(tile) for tile in tiles]
    return ee.ImageCollection(tiled_results).flatten()

In [32]:
print(type(NDVIFiltered))



<class 'ee.featurecollection.FeatureCollection'>


In [31]:
tile_size = 0.5  # tile size in degrees

NDVIFiltered = ndviCALC_step4(NDVIFiltered, SnowCover, window_size=3, region = bounding_box, tile_size = tile_size, target_projection=modis_projection)
print("Step 5 complete: Applied moving window with a window size of 3.")
print_projection_info(NDVIFiltered, "Processed NDVI (Step 5)")
print(type(NDVIFiltered))

NDVIFiltered_floored = ndviCALC_step5(NDVIFiltered, quantile_value=2.5, region = bounding_box, tile_size = tile_size)
print("Step 6 complete: Calculated the floor (0.025 quantile).")
print_image_projection(NDVIFiltered_floored, name="Baseline NDVI Image (Step 6)")
print(type(NDVIFiltered_floored))

NDVIFiltered = ndviCALC_step6(NDVIFiltered, SnowCover, region = bounding_box, tile_size = tile_size)
print("Step 7 complete: Applied baseline flooring and added winter and snow masks.")
print_projection_info(NDVIFiltered, "Processed NDVI (Step 7)")
print(type(NDVIFiltered))


AttributeError: 'Feature' object has no attribute 'clip'

In [ ]:

# # # Get the first image (winter) and the twentieth image (non-winter) for visualization

# modis_projection = modis_ndvi.first().projection()
# modis_scale = 250  # Set to MODIS native resolution, e.g., 500m or 250m

# ## get the images
# i1 = modis_ndvi.toList(modis_ndvi.size()).get(0)
# i1 = ee.Image(i1)
# i1 = i1.reproject(crs=modis_projection)

# i4 = NDVIFiltered.toList(NDVIFiltered.size()).get(29)
# i4 = ee.Image(i4)
# i4 = i4.reproject(crs=modis_projection)

# floor = ee.Image(NDVIFiltered_floored)

# # Define visualization parameters
# vis_params = {
#     'min': 0,
#     'max': 1,
#     'palette': ['blue', 'white', 'green']  
# }

# # Initialize the interactive map
# Map = geemap.Map()

# # Add the original NDVI layer to the map
# Map.addLayer(i1.select('NDVI'), vis_params, 'Raw NDVI')
# Map.addLayer(i4.select('NDVI'), vis_params, 'Filtered NDVI Image 1')

# # Add the corrected NDVI layer for a non-winter month
# # Map.addLayer(twentieth_image.select('NDVI'), vis_params, 'CorrectedNDVI Image 20')

# # Add the original NDVI layer to the map
# # Map.addLayer(first_image2.select('NDVI'), vis_params, 'Filtered NDVI Image 1')

# # Add the corrected NDVI layer for a non-winter month
# # Map.addLayer(twentieth_image2.select('NDVI'), vis_params, 'Filtered NDVI Image 20')

# # Add the winter baseline NDVI
# # Map.addLayer(NDVIFiltered_floored.select('NDVI_baseline'), vis_params, 'Winter Baseline NDVI')

# # Center the map on the bounding box
# Map.centerObject(reprojected_bounding_box)
# Map.addLayerControl()

# # Show the map
# Map


In [6]:
## original bounding box

def create_grid(bounding_box, rows, cols):
    bounds = bounding_box.bounds().getInfo()['coordinates'][0]
    min_lon, min_lat = bounds[0]
    max_lon, max_lat = bounds[2]

    lon_step = (max_lon - min_lon) / cols
    lat_step = (max_lat - min_lat) / rows

    grid = []
    for i in range(rows):
        for j in range(cols):
            lon1 = min_lon + lon_step * j
            lat1 = min_lat + lat_step * i
            lon2 = lon1 + lon_step
            lat2 = lat1 + lat_step
            tile = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
            grid.append(tile)

    return grid

## create a grid of bounding boxes
rows, cols = 5, 5 ## create a 25 tile grid
grid = create_grid(bounding_box, rows, cols)

def print_grid_info(grid):
    # Iterate through each tile in the grid and print its bounds and CRS
    print("Grid Information:")
    for idx, tile in enumerate(grid):
        bounds = tile.bounds(maxError=1).coordinates().getInfo()  # Add maxError
        crs = tile.projection().crs().getInfo()  # Get CRS string from projection
        print(f"Tile {idx + 1}:")
        print(f"  Bounds: {bounds}")
        print(f"  CRS: {crs}")
        print()


Here, we are going to export the image collections and read them back in as rasters.

In [7]:
## function to generate yearly date ranges
def generate_yearly_date_ranges(start_date, end_date):
    start_year = datetime.strptime(start_date, "%Y-%m-%d").year
    end_year = datetime.strptime(end_date, "%Y-%m-%d").year
    yearly_ranges = []

    for year in range(start_year, end_year + 1):
        start_of_year = f"{year}-01-01"
        end_of_year = f"{year}-12-31"
        yearly_ranges.append((start_of_year, end_of_year))

    return yearly_ranges

## generate Yearly Date Ranges
start_date_export = '2018-01-01'
end_date_export = '2023-12-31'

yearly_ranges = generate_yearly_date_ranges(start_date_export, end_date_export)

print("Yearly Ranges:", yearly_ranges)

Yearly Ranges: [('2018-01-01', '2018-12-31'), ('2019-01-01', '2019-12-31'), ('2020-01-01', '2020-12-31'), ('2021-01-01', '2021-12-31'), ('2022-01-01', '2022-12-31'), ('2023-01-01', '2023-12-31')]


In [8]:

# ## function to create a grid of bounding boxes
# def create_grid(bounding_box, rows, cols):
#     bounds = bounding_box.bounds().getInfo()['coordinates'][0]
#     min_lon, min_lat = bounds[0]
#     max_lon, max_lat = bounds[2]

#     lon_step = (max_lon - min_lon) / cols
#     lat_step = (max_lat - min_lat) / rows

#     grid = []
#     for i in range(rows):
#         for j in range(cols):
#             lon1 = min_lon + lon_step * j
#             lat1 = min_lat + lat_step * i
#             lon2 = lon1 + lon_step
#             lat2 = lat1 + lat_step
#             tile = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
#             grid.append(tile)

#     return grid

## create a grid of bounding boxes
# rows, cols = 5, 5 ## create a 25 tile grid
# grid = create_grid(bounding_box, rows, cols)

## export each tile for each year with multiple 8-day layers
def export_yearly_tiles_with_bands(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports'):
    for start_date, end_date in yearly_ranges:
        # Ensure each image is converted to float within the yearly collection
        yearly_NDVI = corrected_NDVI.filterDate(start_date, end_date).map(lambda image: image.select(['NDVI']).toFloat())
        print(f"Exporting data for year: {start_date} to {end_date}")

        # Combine each 8-day period in the year into separate bands
        ndvi_bands = yearly_NDVI.toBands()  # Converts the collection to a multi-band image

        for tile_index, tile in enumerate(grid):
            # Clip each tile's region from the multi-band image
            export_image = ndvi_bands.clip(tile)

            # Define the export task for each tile and year
            export_task = ee.batch.Export.image.toDrive(
                image=export_image,
                description=f'Merkle_NDVI_{start_date[:4]}_Tile_{tile_index + 1}',
                scale=target_scale,  # use the MODIS scale (e.g., 231.656m)
                region=tile,
                crs=target_projection['crs'],  # explicitly specify the projection
                fileFormat='GeoTIFF',
                folder=folder_name
            )
            export_task.start()
            print(f"Started export task for Year {start_date[:4]} Tile {tile_index + 1}")


yearly_ranges = [('2018-01-01', '2018-12-31')]
## perform the export with multi-band export enabled
export_yearly_tiles_with_bands(NDVIFiltered, grid, yearly_ranges, folder_name='NDVI_Exports')




Exporting data for year: 2018-01-01 to 2018-12-31
Started export task for Year 2018 Tile 1
Started export task for Year 2018 Tile 2
Started export task for Year 2018 Tile 3
Started export task for Year 2018 Tile 4
Started export task for Year 2018 Tile 5
Started export task for Year 2018 Tile 6
Started export task for Year 2018 Tile 7
Started export task for Year 2018 Tile 8
Started export task for Year 2018 Tile 9
Started export task for Year 2018 Tile 10
Started export task for Year 2018 Tile 11
Started export task for Year 2018 Tile 12
Started export task for Year 2018 Tile 13
Started export task for Year 2018 Tile 14
Started export task for Year 2018 Tile 15
Started export task for Year 2018 Tile 16
Started export task for Year 2018 Tile 17
Started export task for Year 2018 Tile 18
Started export task for Year 2018 Tile 19
Started export task for Year 2018 Tile 20
Started export task for Year 2018 Tile 21
Started export task for Year 2018 Tile 22
Started export task for Year 2018 T